In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd

# Add project root directory to Python path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [2]:
# From scripts/
from scripts.protein_datamodule import ProteinSequenceDataset, ProteinDataModule
from scripts.utils import train_model, evaluate_model, plot_training_history

# From models/
from models import LSTMClassifier, VanillaRNNClassifier

In [3]:
df = pd.read_csv("../data/finalized_dataset.csv")
display(df)

,Unnamed: 0.1,Unnamed: 0,uniProtkbId,proteinName,sequence,subcellularLocationSingle
0,0,0,CP2D7_HUMAN,Putative cytochrome P450 2D7,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,Cytoplasm
1,1,2,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c,MRWQEMGYIFYPRKLR,Nucleus
2,2,7,RBM47_HUMAN,RNA-binding protein 47,MTAEDSTAAMSSDSAAGSSAKVPEGVAGAPNEAALLALMERTGYSM...,Nucleus
3,3,7,RBM47_HUMAN,RNA-binding protein 47,MTAEDSTAAMSSDSAAGSSAKVPEGVAGAPNEAALLALMERTGYSM...,Cytoplasm
4,4,10,E2F8_HUMAN,Transcription factor E2F8,MENEKENLFCEPHKRGLMKTPLKESTTANIVLAEIQPDFGPLTTPT...,Nucleus
...,...,...,...,...,...,...
13481,13481,20053,ZN706_HUMAN,Zinc finger protein 706,MARGQQKIQSQQKNAKKQAGQKKKQGHDQKAAAKAALIYTCTVCRT...,Cytoplasm
13482,13482,20053,ZN706_HUMAN,Zinc finger protein 706,MARGQQKIQSQQKNAKKQAGQKKKQGHDQKAAAKAALIYTCTVCRT...,Nucleus
13483,13483,20054,SNURF_HUMAN,SNRPN upstream reading frame protein,MERARDRLHLRRTTEQHVPEVEVQVKRRRTASLSNQECQLYPRRSQ...,Nucleus
13484,13484,20056,HMN13_HUMAN,Humanin-like 13,MDTQGFSCLLLLISEIDLSVKRRI,Cytoplasm


In [4]:
# instantiate once
csv_path = "../data/finalized_dataset.csv"
datamodule = ProteinDataModule(csv_path, batch_size=128)

In [5]:
# grab the very first sample from the train‐set
toks, lbl = datamodule.train_ds[0]

print("TOKENS:", toks)      # a 1D LongTensor of token indices
print("LABEL :", lbl)       # a single integer

# if you want to turn it back into the AA‐string:
idx2aa = {v:k for k,v in datamodule.aa2idx.items()}
recon_seq = "".join(idx2aa[i.item()] for i in toks)
print("RECONSTRUCTED SEQ:", recon_seq)

# and to get the human‐readable label name back:
idx2label = {v:k for k,v in datamodule.label2idx.items()}
print("LABEL NAME        :", idx2label[lbl])


TOKENS: tensor([10, 11, 12,  0, 11,  6, 15, 13, 17,  0,  5,  4, 17,  9,  9,  5,  9, 15,
        13, 17, 18,  3,  9, 14,  4, 17,  4,  4, 16, 17,  4, 15,  0, 17, 19,  4,
        10, 16, 17, 17,  5, 11,  9,  9,  7, 17, 17,  7, 17, 16, 15,  2, 12,  6,
         9,  6, 16, 16, 10, 19,  4,  9,  9,  5, 11,  9, 15,  4,  9,  2,  4,  1,
        19, 15, 15,  7, 16,  0, 12, 14, 10,  9, 17,  2,  9,  9, 15,  5, 11, 12,
        16,  7, 15,  4,  5,  5,  1,  9, 16, 13,  9,  4,  4,  4,  6,  4,  7,  5,
         5,  7,  8,  7,  4,  9,  9, 16, 17, 10,  0, 19,  2, 14, 19,  7,  0,  7,
        15, 13, 12,  9,  6, 19, 16,  9,  7, 10, 11, 13, 16, 17,  1,  0,  9,  9,
        10,  0,  0, 15, 18, 17,  5,  5,  4,  7,  6, 15,  7, 17, 13,  7,  0,  9,
        16,  7, 13,  9, 12,  4,  1,  5, 12,  2,  8,  9,  2, 11,  4, 19,  1,  2,
        17, 12, 13,  9,  7,  8,  9,  0,  1, 16,  2, 16,  4, 17,  9,  3,  9,  9,
        10, 17, 15, 11, 11,  5,  9, 17, 16,  9, 10,  1,  4,  9, 17,  9,  9,  5,
        15, 19, 16,  0,  9,  9, 

<h3>Training and Evaluate the model</h3>

In [ ]:
# Initialize Model
model = LSTMClassifier(
    vocab_size=22,  # 20 AAs + X + <pad>
    embed_dim=128,
    hidden_dim=256,
    num_layers=2,
    num_classes=len(datamodule.label2idx),
    bidir=True
)

# Train
history = train_model(model, datamodule)

# Plot
plot_training_history(history)

Epoch 1:   0%|          | 0/85 [00:00<?, ?it/s]

Epoch 1/10: Train Loss: 1.0056, Acc: 0.4831 | Val Loss: 1.0102, Acc: 0.4915


Epoch 2:   0%|          | 0/85 [00:00<?, ?it/s]

Epoch 2/10: Train Loss: 0.9986, Acc: 0.4737 | Val Loss: 0.8290, Acc: 0.5982


Epoch 3:   0%|          | 0/85 [00:00<?, ?it/s]

<h3>Plotting Training and Validation Accuracy and Loss</h3>